In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import glob
import json
import secrets
# import networkx as nx
# import matplotlib.pyplot as plt


#### Linear and Nonlinear

In [ ]:
np.random.seed(42)
T = 1000
maxlag = 3
x = np.random.randn(T + maxlag, 11)
for t in range(maxlag, T + maxlag):
    x[t, 0] += 0.9 * x[t - 1, 0]
    x[t, 1] += 0.6 * x[t, 1] + 0.5 * x[t - 2, 0]
    x[t, 2] += 0.4 * x[t, 2] - 0.4 * x[t - 3, 0] + 0.3 * x[t - 1, 1]

    x[t, 3] += 0.25 * x[t - 1, 3]
    x[t, 4] += 0.25 * x[t - 1, 4] - 0.25 * x[t - 1, 3]
    x[t, 5] += 0.25 * x[t - 1, 5] - 0.50 * x[t - 2, 3] - 0.50 * x[t - 2, 4]
    x[t, 6] += 0.25 * x[t - 1, 6] - 0.25 * x[t - 3, 3] + 0.25 * x[t - 3, 4]
    x[t, 7] += 0.25 * x[t - 1, 7] + 0.25 * x[t - 1, 3] + 0.25 * x[t - 2, 4]
    x[t, 8] += 0.25 * x[t - 1, 8] + 0.25 * x[t - 1, 3] + 0.25 * x[t - 2, 7]
    x[t, 9] += 0.25 * x[t - 1, 9] + 0.25 * x[t - 1, 4] + 0.25 * x[t - 2, 8]
    x[t, 10] += 0.25 * x[t - 1, 10] + 0.25 * x[t - 1, 4] + 0.25 * x[t - 2, 9]

df_linear = pd.DataFrame(x[maxlag:], columns=["X" + str(i) for i in range(x.shape[1])])
df_linear.max()

# df_linear.tail(1000).plot()
df_linear.corr()


In [ ]:
np.random.seed(42)
T = 500
maxlag = 3
x = np.random.randn(T + maxlag, 11)
for t in range(maxlag, T + maxlag):
    x[t, 0] += 0.9 * x[t - 1, 0]  # Plcg
    x[t, 1] += 0.6 * x[t, 1] + 0.5 * x[t - 2, 0]  # PIP3
    x[t, 2] += 0.4 * x[t, 2] - 0.4 * x[t - 3, 0] + 0.3 * x[t - 1, 1]  # PIP2

    x[t, 3] += 0.25 * x[t - 1, 3]  # PKC
    x[t, 4] += 0.25 * x[t - 1, 4] - 0.25 * x[t - 1, 3]  # PKA
    x[t, 5] += 0.25 * x[t - 1, 5] + 0.50 * x[t - 2, 3] - 0.50 * x[t - 1, 4]  # P38
    x[t, 6] += 0.30 * x[t - 1, 6] - 0.25 * x[t - 3, 3] + 0.25 * x[t - 3, 4]  # Jnk
    x[t, 7] += 0.40 * x[t - 1, 7] + 0.35 * x[t - 1, 3] + 0.25 * x[t - 2, 4]  # Raf
    x[t, 8] += (
        0.50 * x[t - 1, 8] - 0.5 * x[t - 1, 3] + 0.25 * x[t - 1, 4] + 0.6 * x[t - 2, 7]
    )  # Mek
    x[t, 9] += 0.6 * x[t - 1, 9] + 0.45 * x[t - 1, 4] - 0.35 * x[t - 2, 8]  # Erk
    x[t, 10] += 0.7 * x[t - 1, 10] - 0.25 * x[t - 1, 4] + 0.25 * x[t - 2, 9]  # Akt
var_names = [
    "Plcg",
    "Pip3",
    "Pip2",
    "PKC",
    "PKA",
    "P38",
    "Jnk",
    "Raf",
    "Mek",
    "Erk",
    "Akt",
]
df = pd.DataFrame(x[maxlag:], columns=var_names)

In [ ]:
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4196918/ (2014)
# https://eurasip.org/Proceedings/Eusipco/Eusipco2020/pdfs/0002388.pdf (different formulation!)
np.random.seed(42)
T = 100
maxlag = 3
x = np.random.randn(T + maxlag, 5)
for t in range(maxlag, T + maxlag):
    x[t, 0] += 0.95 * np.sqrt(2.0) * x[t - 1, 0] - 0.9025 * x[t - 2, 0]
    x[t, 1] += 0.5 * x[t - 2, 0]  # X0 -> X1
    x[t, 2] += -0.4 * x[t - 3, 0]  # X0 -> X2
    x[t, 3] += (
        0.25 * np.sqrt(2.0) * x[t - 1, 3]
        - 0.50 * x[t - 2, 0]
        + 0.25 * np.sqrt(2.0) * x[t - 1, 4]
    )  # X0 -> X2
    x[t, 4] += 0.25 * np.sqrt(2.0) * x[t - 1, 4] - 0.25 * np.sqrt(2.0) * x[t - 1, 3]
df_linear = pd.DataFrame(x[maxlag:], columns=["X" + str(i) for i in range(x.shape[1])])
df_linear

x = np.random.randn(T + maxlag, 5)
for t in range(maxlag, T + maxlag):
    x[t, 0] += 0.95 * np.sqrt(2.0) * x[t - 1, 0] - 0.9025 * x[t - 2, 0]
    x[t, 1] += 0.5 * x[t - 2, 0] ** 2.0  # X0 -> X1
    x[t, 2] += -0.4 * x[t - 3, 0]  # X0 -> X2
    x[t, 3] += (
        0.25 * np.sqrt(2.0) * x[t - 1, 3]
        - 0.50 * x[t - 2, 0] ** 2
        + 0.25 * np.sqrt(2.0) * x[t - 1, 4]
    )  # X0 -> X3 & X4-> X3
    x[t, 4] += (
        0.25 * np.sqrt(2.0) * x[t - 1, 4] - 0.25 * np.sqrt(2.0) * x[t - 1, 3]
    )  # X3 -> X4
df_nl = pd.DataFrame(
    x[maxlag:], columns=["X" + str(i) for i in range(x.shape[1])]
)  # X4 -> X3
df_nl


In [ ]:
np.random.seed(42)
T = 500
C = np.arange(T).reshape(-1, 1)
x3 = np.random.randn(T, 1) + 3 * (C**2 / 100000)
x1 = 0.8 * x3 + 3 * (C**2 / 100000) + 0.5 * np.random.randn(T, 1)
x4 = -0.8 * x3 + (C**2 / 100000) + 0.5 * np.random.randn(T, 1)
x2 = 0.5 * x4 + 0.5 * x1 + 2 * (C**2 / 100000) + 0.5 * np.random.randn(T, 1)
data = pd.DataFrame(
    np.concatenate([C, x1, x2, x3, x4], -1), columns=["C", "X1", "X2", "X3", "X4"]
)
data

#### Henon Maps (Chaos)

In [ ]:
# https://eurasip.org/Proceedings/Eusipco/Eusipco2020/pdfs/0002388.pdf  (is it really Henon?)
# https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0194382 (different version)
T = 200
M = 5
Q = 0.5  # Coupling Strenght
maxlag = 2
x = np.random.randn(T + maxlag, M)
for t in range(maxlag, T + maxlag):
    for m in range(M):
        if m in [0, M - 1]:
            x[t, m] = 1.4 - x[t - 1, m] ** 2 + 0.3 * x[t - 2, m]
        else:
            x[t, m] = (
                1.4
                - (
                    0.5
                    * Q
                    * (x[t - 1, m - 1] + x[t - 1, m + 1] + (1 - Q) * x[t - 1, m])
                )
                ** 2
                + 0.3 * x[t - 2, m]
            )
x[maxlag:, :].shape

In [ ]:
## Original
def henon_attractor(x, y, a=1.4, b=0.3):
    """Computes the next step in the Henon
    map for arguments x, y with kwargs a and
    b as constants.
    """
    x_next = 1 - a * x**2 + y
    y_next = b * x
    return x_next, y_next


# number of iterations and array initialization
steps = 100
X = np.zeros(steps + 1)
Y = np.zeros(steps + 1)

# starting point
X[0], Y[0] = 0, 0

# add points to array
for i in range(steps):
    x_next, y_next = henon_attractor(X[i], Y[i])
    X[i + 1] = x_next
    Y[i + 1] = y_next

### Generate Random Graphs

> **Note:**  
> The following code was used to generate synthetic data in the paper.  
> However, the original random seed was not saved, so it may not be possible to exactly replicate the graphs and data used in the publication.

In [ ]:
from .structural_causal_processes import (
    structural_causal_process,
    generate_structural_causal_process,
    check_stationarity,
)

Root_DIR = os.path.join(os.getcwd(), "..")
sys.path.append(Root_DIR)
SIM_DIR = os.path.join(Root_DIR, "data/simulated_autoregressive")


In [ ]:
# import secrets

dependency_coeffs = list(np.arange(40, 90, 1) / 100) + list(
    np.arange(-40, -90, -1) / 100
)
auto_coeffs = list(np.arange(30, 80, 1) / 100)
np.random.shuffle(dependency_coeffs)


def lin(x):
    return x


def nl1(x):
    return x + 0.5 * x**2


def nl2(x):
    return x + np.sin(0.1 * x)


# def nl3(x): return (np.exp(-x**2 / 20.))
dependency_funcs = ["lin", "lin", "nl1", "nl2"]
func_map = {"lin": lin, "nl1": nl1, "nl2": nl2}

## Generate Data
for nVars in [3, 4, 5, 6, 8, 10, 15]:
    T = 3000  ## to make sure any shorter time series is (almost) stationary
    max_lag = 5
    min_sparsity = 0.45
    total_links = nVars * (nVars - 1) / 2
    # lb = max(1,nVars-2)
    lb = int(nVars - 1.9 - 0.009 * nVars**2)
    ub = np.round(2.25 * nVars - 4)

    counter = 0

    while counter < 50:
        nLinks = np.random.randint(lb, ub + 1)
        links, noises = generate_structural_causal_process(
            N=nVars,
            L=nLinks,
            max_lag=max_lag,
            dependency_funcs=dependency_funcs,
            auto_coeffs=auto_coeffs,
            dependency_coeffs=dependency_coeffs,
            seed=None,
        )

        if check_stationarity(links):
            data, nonstat = structural_causal_process(
                links, T=T, noises=noises, func_map=func_map
            )
            if not nonstat:
                print(nVars, counter, "  ", end="\r")
                total_links = sum([len(links[k]) - 1 for k in links.keys()])
                funcs = []
                for k in links.keys():
                    for j in range(1, len(links[k])):
                        funcs.append(links[k][j][2])  # .__str__().split(' ')[1])
                funcs = "_".join(
                    [
                        str(v) + k
                        for k, v in pd.Series(funcs).value_counts().to_dict().items()
                    ]
                )

                fname = f"SCM-N{nVars}-L{nLinks}-{funcs}-lags{max_lag}-{secrets.token_hex(nbytes=8)}.json"

                with open(os.path.join(SIM_DIR, fname), "w") as outfile:
                    json.dump(links, outfile)
                counter += 1
    # else:
    # data, nonstat = structural_causal_process(links,T=T, noises=noises,func_map=func_map)
    # print('non_stationary',nonstat)


##### Generate data Files

In [ ]:
sys.path.append("..")

sims_path = "../data/simulated_autoregressive/"
dir_list = glob.glob("../data/simulated_autoregressive/*.json")
# np.random.seed(42) ## The seed for the paper was not set!, in future lets set the seed

for T in [300, 500, 1000, 2000]:
    for j, gf in enumerate(dir_list):
        jFile = dir_list[j]
        fname = jFile.replace(".json", "")
        links = json.load(open(jFile))
        links = {int(k): v for k, v in links.items()}
        data, nonstat = structural_causal_process(
            links, T=T, noises=None, func_map=func_map
        )
        pd.DataFrame(data, columns=[f"X{i}" for i in range(data.shape[1])]).to_parquet(
            fname + f"-T{T}.parquet"
        )


np.random.seed(42)
for T in [50, 150]:
    for j, gf in enumerate(dir_list):
        jFile = dir_list[j]
        fname = jFile.replace(".json", "")
        links = json.load(open(jFile))
        links = {int(k): v for k, v in links.items()}
        data, nonstat = structural_causal_process(
            links, T=T, noises=None, func_map=func_map
        )
        pd.DataFrame(data, columns=[f"X{i}" for i in range(data.shape[1])]).to_parquet(
            fname + f"-T{T}.parquet"
        )